# Google Colab: Qwen Worker for Remote AnyServe

This notebook assumes `anyserve` is already deployed somewhere else.

1. load `Qwen/Qwen3-0.6B` from Hugging Face with `transformers`
2. expose it as a tiny OpenAI-compatible upstream on `127.0.0.1:8000`
3. start only the `anyserve worker` process inside Colab
4. connect that worker to a remote AnyServe gRPC endpoint
5. optionally call the already-deployed AnyServe public gateway URL directly

Notes:

- this notebook is chat-only
- the deployed AnyServe gateway should already expose `Qwen/Qwen3-0.6B` in `[openai].models`
- model aliasing is not implemented yet, so the gateway model name should match exactly
- Colab must be able to reach the remote gRPC endpoint over the network
- this notebook does not start a local AnyServe gateway and does not use Cloudflare Tunnel


In [ ]:
%pip install -q fastapi uvicorn transformers accelerate sentencepiece requests
!curl -fsSL https://raw.githubusercontent.com/anyserve/anyserve/main/scripts/install.sh | sh -s -- --dir /usr/local/bin


In [ ]:
import os
import pathlib
import textwrap

MODEL_ID = os.environ.get("HF_MODEL_ID", "Qwen/Qwen3-0.6B")
UPSTREAM_PORT = 8000
REMOTE_ANYSERVE_GRPC_ENDPOINT = os.environ.get(
    "REMOTE_ANYSERVE_GRPC_ENDPOINT",
    "https://your-anyserve-grpc.example.com",
)
REMOTE_ANYSERVE_PUBLIC_BASE_URL = os.environ.get(
    "REMOTE_ANYSERVE_PUBLIC_BASE_URL",
    "https://your-anyserve-gateway.example.com",
)

WORKDIR = pathlib.Path("/content/anyserve-colab-remote")
WORKDIR.mkdir(parents=True, exist_ok=True)

(WORKDIR / "worker.toml").write_text(
    textwrap.dedent(
        f"""
        base_url = "http://127.0.0.1:{UPSTREAM_PORT}/v1"
        provider = "huggingface-transformers"
        interfaces = ["llm.chat.v1"]
        poll_interval_secs = 1
        heartbeat_interval_secs = 5
        upstream_timeout_secs = 120

        [attributes]
        region = "colab"

        [capacity]
        slot = 1
        """
    ).strip()
    + "\n"
)

hf_server = '''
import os
import time
import uuid
from typing import Any, Dict, List

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("HF_MODEL_ID", "Qwen/Qwen3-0.6B")
PORT = int(os.environ.get("PORT", "8000"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

app = FastAPI()


class ChatCompletionRequest(BaseModel):
    model: str | None = None
    messages: List[Dict[str, Any]]
    max_tokens: int | None = 256
    temperature: float | None = 0.7
    top_p: float | None = 0.95
    stream: bool | None = False


def render_prompt(messages: List[Dict[str, Any]]) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            try:
                return tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                return tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
        except Exception:
            pass

    lines = []
    for message in messages:
        role = message.get("role", "user")
        content = message.get("content", "")
        if isinstance(content, list):
            chunks = []
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    chunks.append(item.get("text", ""))
            content = "\n".join(chunks)
        lines.append(f"{role}: {content}")
    lines.append("assistant:")
    return "\n".join(lines)


@app.get("/healthz")
def healthz() -> Dict[str, Any]:
    return {"ok": True, "model": MODEL_ID}


@app.get("/v1/models")
def list_models() -> Dict[str, Any]:
    created = int(time.time())
    return {
        "object": "list",
        "data": [
            {
                "id": MODEL_ID,
                "object": "model",
                "created": created,
                "owned_by": "huggingface",
            }
        ],
    }


@app.post("/v1/chat/completions")
def chat(request: ChatCompletionRequest) -> Dict[str, Any]:
    if request.stream:
        raise HTTPException(
            status_code=400,
            detail="stream=true is not implemented in this notebook example",
        )

    if request.model and request.model != MODEL_ID:
        raise HTTPException(
            status_code=400,
            detail=f"this notebook only serves {MODEL_ID}",
        )

    prompt = render_prompt(request.messages)
    batch = tokenizer(prompt, return_tensors="pt")
    batch = {name: tensor.to(model.device) for name, tensor in batch.items()}

    do_sample = (request.temperature or 0.0) > 0
    max_tokens = request.max_tokens or 256

    generate_kwargs = {
        **batch,
        "max_new_tokens": max_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generate_kwargs["temperature"] = request.temperature if request.temperature is not None else 0.7
        generate_kwargs["top_p"] = request.top_p if request.top_p is not None else 0.95

    with torch.inference_mode():
        output_ids = model.generate(**generate_kwargs)

    new_tokens = output_ids[0, batch["input_ids"].shape[1] :]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    created = int(time.time())

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": created,
        "model": MODEL_ID,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": int(batch["input_ids"].shape[1]),
            "completion_tokens": int(new_tokens.shape[0]),
            "total_tokens": int(batch["input_ids"].shape[1] + new_tokens.shape[0]),
        },
    }


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
'''

(WORKDIR / "hf_openai_server.py").write_text(textwrap.dedent(hf_server).strip() + "\n")

print(f"workdir: {WORKDIR}")
print(f"remote grpc endpoint: {REMOTE_ANYSERVE_GRPC_ENDPOINT}")
print(f"remote public base url: {REMOTE_ANYSERVE_PUBLIC_BASE_URL}")
print((WORKDIR / "worker.toml").read_text())


In [ ]:
import os
import subprocess
import time

import requests

processes = {}


def start_process(name, cmd, extra_env=None):
    log_path = WORKDIR / f"{name}.log"
    proc = subprocess.Popen(
        cmd,
        cwd=WORKDIR,
        env={**os.environ, **(extra_env or {})},
        stdout=log_path.open("w"),
        stderr=subprocess.STDOUT,
        start_new_session=True,
        text=True,
    )
    processes[name] = {"proc": proc, "log": log_path}
    print(f"started {name}: pid={proc.pid}, log={log_path}")
    return proc


def wait_http(url, timeout=180):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=5)
            if response.ok:
                return response
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    raise RuntimeError(f"timed out waiting for {url}: {last_error}")


def tail(name, lines=40):
    log_path = processes[name]["log"]
    text = log_path.read_text() if log_path.exists() else ""
    print("\n".join(text.splitlines()[-lines:]))


start_process(
    "hf-server",
    ["python", "hf_openai_server.py"],
    {"HF_MODEL_ID": MODEL_ID, "PORT": str(UPSTREAM_PORT)},
)
wait_http(f"http://127.0.0.1:{UPSTREAM_PORT}/healthz", timeout=900)
requests.get(f"http://127.0.0.1:{UPSTREAM_PORT}/v1/models", timeout=30).json()


In [ ]:
if "your-anyserve" in REMOTE_ANYSERVE_GRPC_ENDPOINT:
    raise ValueError(
        "set REMOTE_ANYSERVE_GRPC_ENDPOINT to the deployed AnyServe gRPC endpoint first"
    )

start_process(
    "anyserve-worker",
    [
        "anyserve",
        "--endpoint",
        REMOTE_ANYSERVE_GRPC_ENDPOINT,
        "worker",
        "--config",
        "worker.toml",
    ],
)
time.sleep(8)
tail("anyserve-worker", lines=60)


In [ ]:
if "your-anyserve-gateway" in REMOTE_ANYSERVE_PUBLIC_BASE_URL:
    print("Set REMOTE_ANYSERVE_PUBLIC_BASE_URL to run the remote gateway smoke test.")
else:
    wait_http(f"{REMOTE_ANYSERVE_PUBLIC_BASE_URL}/readyz", timeout=120)
    print(requests.get(f"{REMOTE_ANYSERVE_PUBLIC_BASE_URL}/v1/models", timeout=30).json())

    response = requests.post(
        f"{REMOTE_ANYSERVE_PUBLIC_BASE_URL}/v1/chat/completions",
        json={
            "model": MODEL_ID,
            "messages": [
                {
                    "role": "user",
                    "content": "Reply with one short sentence saying this request went through a remote AnyServe deployment.",
                }
            ],
        },
        timeout=300,
    )
    response.raise_for_status()
    print(response.json())

    print("\nShare this curl with another person:\n")
    print(
        textwrap.dedent(
            f"""
            curl {REMOTE_ANYSERVE_PUBLIC_BASE_URL}/v1/chat/completions \\
              -H 'content-type: application/json' \\
              -d '{{"model": "{MODEL_ID}", "messages": [{{"role": "user", "content": "hello from another laptop"}}]}}'
            """
        ).strip()
    )


In [ ]:
import signal

for name, info in processes.items():
    proc = info["proc"]
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        print(f"stopped {name}")
